In [ ]:
from experiments.plots.paper.rd_mspe_wz import conditional_rd_mape
import gzip
import pickle
import numpy as np

path_to_file =lambda agent_id, round_id: f'D:\\User\\App Files\\Projects\\VUB-ACS-25_Thesis\\'+\
                                         f'experiments\\run_sim_script\\past recons list\\w{agent_id+1}\\round_{round_id}.pkl.gz'

list_si = [[] for i in range(5)]
for agent_id in range(5):
    for round_id in range(5):
        with gzip.open(path_to_file(agent_id, round_id), 'rb') as f:
            recons_vector, dict_shape = pickle.load(f)
        list_si[agent_id].append(recons_vector)
print(len(list_si[0]))

i=0
target=list_si[0][-i]
si = [*list_si[0][-10-i:-i],
      *list_si[1][-10-i:],
      *list_si[2][-10-i:],
      *list_si[3][-10-i:],
      *list_si[4][-10-i:],]
si = np.array(si).T

In [ ]:
rand_id = np.random.permutation(len(target))#[:1_000_000]
si = si[rand_id]
target = target[rand_id]

In [ ]:
side_info_data = [a for a in si.T]
y = target

In [ ]:
from components.broadcast_components.WZ_models.wz_quant_RNN import PL_EncoderDecoder_RNN
class SignQ:
    def __init__(self):
        self.recons_center = None

    def basic_encoding(self, grad_vector):
        temp = (grad_vector >= 0)
        indices = temp.astype(int)
        self.recons_center = [grad_vector[~temp].mean(), grad_vector[temp].mean()]
        return indices

    def basic_decoding(self, quantized_data):
        assert self.recons_center is not None
        res = np.array([self.recons_center[i] for i in quantized_data])
        return res

class wz_model_class(PL_EncoderDecoder_RNN):
    def get_prior_and_softcodes_net(self, grad_vector, side_info=None):
        assert not self.coding_model.training
        assert self.coding_model.marginal == (side_info is None or len(side_info)==0)

        kir = SignQ()
        hard_codes = torch.tensor(kir.basic_encoding(grad_vector.cpu().numpy()), device=grad_vector.device).T[0]
        soft_codes = [torch.stack([1-hard_codes, hard_codes]).T]
        priors = self.coding_model.get_priors(codes=soft_codes, y=side_info, tau=None)

        return torch.stack(priors), torch.stack(soft_codes)

In [ ]:
import torch
from components.broadcast_components.WZ_models.Learned_non_wz_quantizer import LearnedNonWZQuantizer
from components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol import fix_zero_probabilities

wz_model = wz_model_class(inp_dim=1, side_info_size=0, num_planes=1, bins_per_plane=2, tau=1.5,
                                 tau_rate=10, reconst_ld=400, lr=1e-3, marginal=False).to(torch.float32)
wz_quantizer = LearnedNonWZQuantizer(wz_model, train_sample_size=200_000, count_side_info_data=1,
                                enable_progress_bar=True, vec_slices=[])
wz_quantizer.train_model(y, side_info_data)
bins,temp = wz_quantizer.encoding_process(y)
recons = wz_quantizer.decoding_process(bins, [], temp)
prior = wz_quantizer.get_set_training_posterior_cdf()
prior = fix_zero_probabilities(prior, bins)

In [ ]:
kir = SignQ()
bins = [kir.basic_encoding(y)]
recons = kir.basic_decoding(bins[0])

In [ ]:
from components.broadcast_components.compressor.rans_coding import rans_batch_encode
from components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol import compress_data_list
from components.broadcast_components.broadcasting_process.broadcast_reporting_utilities import get_obj_size

temp = [rans_batch_encode(b, p) for b, p in zip(bins, prior)]
rate_emp = get_obj_size(compress_data_list(temp)) / len(y) * 8
rate_th = sum([p[torch.arange(len(b)), b] for b, p in zip(bins, prior)])

print("avg bit rate:", rate_emp)
print("avg bit rate (theo):", np.mean(-np.log2(rate_th + 1e-12)))
print("mse:", np.mean((y - recons) ** 2))
print("mape:", np.mean(np.abs(y - recons)) / np.mean(np.abs(y)))

print("with outlier bit rate:", rate_emp)
print("with outlier bit rate (theo):", np.mean(-np.log2(rate_th + 1e-12)))


In [ ]:
res = conditional_rd_mape(x=target, y=si, n_buckets=8, n_x_bins=56, n_recon=56,mape_percent=True,
                          s_grid=np.logspace(-3,3,20), D_targets=np.array([0.05, 0.4]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

D = res["D"]
R = res["R"]

# sort by D so the curve is monotone left→right
idx = np.argsort(D)
plt.plot(D[idx], R[idx])
plt.xlabel("Distortion D (MAPE%)")
plt.ylabel("Rate R (bits/sample)")
plt.title("Empirical lower WZ bound R_{X|Y}(D)")
plt.grid(True)
plt.show()

In [ ]:
# save the curve {D,R} to file

np.savez_compressed('wz_bound.npz', D=D, R=R)
# np.load('wz_bound.npz')
